# QLoRA Supervised Fine-Tuning - Llama-3-8B-Instruct (TechQA)

**Διπλωματικη εργασια - Φαση 2: Learned Abstention & Grounded Resolution**

Το notebook εκπαιδευει το `Meta-Llama-3-8B-Instruct` με **4-bit QLoRA** πανω στο SFT dataset
(450 παραδειγματα, 55% solve / 45% abstain) που παρηχθη απο το RAG pipeline. Στοχος **δεν** ειναι
νεα γνωση αλλα **συμπεριφορα**: σταθερο format (Root cause -> βηματα-> `Technote: [id]`),
πειθαρχια αποχης (MODE B), λιγοτερα εφευρεμενα IDs - δηλαδη ακριβως το *hazard* που το
heuristic retrieval gate δεν μπορουσε να λυσει.

**Ροη:** setup -> φορτωση dataset -> επαληθευση μηκους & masking -> 4-bit load + LoRA -> training ->
merge.

**Απαιτησεις Kaggle:** Accelerator = **GPU T4** (οχι P100). Επισυναψε το `sft_dataset.jsonl`
ως Kaggle Dataset και προσαρμόζουμε το `DATASET_PATH` παρακατω.

## 1. Παραμετροι

In [ ]:
# --- Μοντελο & δεδομενα ---
MODEL_NAME   = "NousResearch/Meta-Llama-3-8B-Instruct"   # ungated mirror - ΙΔΙΑ βαρη με το served μοντελο (llama3:8b-instruct)
DATASET_PATH = "/kaggle/input/techqa-sft/sft_dataset.jsonl"

# --- Output paths ---
OUTPUT_DIR  = "/kaggle/working/llama3-techqa-qlora"
ADAPTER_DIR = "/kaggle/working/adapter"
MERGED_DIR  = "/kaggle/working/merged-fp16"

# --- Ακολουθια ---
MAX_LENGTH  = 4096       # καλυπτει το max μηκος ΧΩΡΙΣ truncation (επαληθευεται στο cell 5)

# --- LoRA ---
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

# --- Training ---
EPOCHS     = 3
LR         = 2e-4        # τυπικο QLoRA  adapters θελουν υψηλοτερο LR απο full-FT
BATCH      = 1
GRAD_ACCUM = 8           # effective batch = BATCH * GRAD_ACCUM = 8
SEED       = 42

## 2. Φορτωση dataset & train/val split (90/10)

In [ ]:
import json
from datasets import Dataset

rows = []
with open(DATASET_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

ABSTAIN = "does not contain a confident solution"
n_b = sum(1 for r in rows if ABSTAIN in r["messages"][-1]["content"])
print("συνολικα παραδειγματα:", len(rows))
print("MODE A (solve):", len(rows) - n_b, " | MODE B (abstain):", n_b,
      f" ({100*n_b/len(rows):.0f}% abstain)")

ds = Dataset.from_list(rows)
split = ds.train_test_split(test_size=0.10, seed=SEED)
train_ds, eval_ds = split["train"], split["test"]
print("\ntrain:", len(train_ds), " | eval:", len(eval_ds))

## 3. Tokenizer

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token   # το padding γινεται mask ουτως η αλλως
tokenizer.padding_side = "right"
print("pad:", tokenizer.pad_token, "| eos:", tokenizer.eos_token)

## 4. Επαληθευση μηκους ακολουθιας (πραγματικος tokenizer)

Το `target` βρισκεται στο **τελος** της ακολουθιας. Αν το `MAX_LENGTH` ειναι μικροτερο απο το
πραγματικο μηκος, η right-truncation **κοβει το target** -> καταστρεφεται το completion-only SFT.
Αυτο το cell μετραει την πραγματικη κατανομη και επιβεβαιωνει οτι κανενα δειγμα δεν περικοπτεται.

In [ ]:
import numpy as np

def _ntok(msgs):
    out = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=False)
    ids = out["input_ids"] if isinstance(out, dict) else out   # νεες εκδοσεις επιστρεφουν dict
    return len(ids)

lengths = np.array([_ntok(r["messages"]) for r in rows])
for p in [50, 90, 95, 99, 100]:
    print(f"p{p:<3}: {int(np.percentile(lengths, p)):>5} tokens")

over = int((lengths > MAX_LENGTH).sum())
print(f"\nδειγματα > MAX_LENGTH({MAX_LENGTH}): {over} / {len(lengths)} ({100*over/len(lengths):.1f}%)")
if over == 0:
    print("OK - κανενα truncation το target μενει ακεραιο σε ολα τα δειγματα.")
else:
    print("[ΠΡΟΣΟΧΗ] καποια δειγματα ξεπερνουν το MAX_LENGTH. Αυξησε το MAX_LENGTH,")

## 5. Φορτωση μοντελου σε 4-bit (NF4)

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # T4: float16 (ΟΧΙ bf16)
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},        # ολο το μοντελο στη GPU 0 (αποφυγη sharding σε 2×T4)
    dtype=torch.float16,       # transformers 5.x: 'dtype' (οχι 'torch_dtype') - αλλιως φορτωνει bf16
)
model.config.use_cache = False   # απαραιτητο με gradient checkpointing
print("Το μοντελο φορτωθηκε σε 4-bit.")

## 6. Διαμορφωση LoRA

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],   # ολες οι γραμμικες προβολες
)

## 7. SFTConfig & SFTTrainer

In [ ]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_length=MAX_LENGTH,
    assistant_only_loss=True,     # loss ΜΟΝΟ στο assistant — mask system+user (Llama-3 template auto-patched)
    packing=False,                # ασυμβατο με σταθερο masking
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    optim="paged_adamw_8bit",
    fp16=True, bf16=False,        # T4
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    activation_offloading=True,   # κατεβαζει τα activations στη CPU RAM — εξοικονομηση GPU μνημης (T4 16GB)
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    seed=SEED,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    peft_config=peft_config,
    processing_class=tokenizer,
)

## 8. Fix precision & Training

Πριν το training, κανουμε cast **ολα τα trainable (LoRA) params σε fp32**. Με fp16 mixed-precision
στον T4, τα master weights πρεπει να ειναι fp32· αν μεινουν bf16, ο fp16 GradScaler σκαει με
`_amp_foreach_non_finite_check_and_unscale_cuda not implemented for BFloat16`. Ο κανονας:
βαση παγωμενη σε 4-bit, adapters fp32, compute fp16 μεσω autocast.

In [ ]:
# Cast trainable (LoRA) params -> fp32 (bulletproof fix για τον fp16 GradScaler στον T4)
n = 0
for _name, p in trainer.model.named_parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.float()
        n += 1
print(f"cast {n} trainable tensors -> fp32")

# επιβεβαιωση: κανενα trainable param δεν ειναι bf16
bad = [nm for nm, p in trainer.model.named_parameters()
       if p.requires_grad and p.dtype == torch.bfloat16]
assert not bad, f"Υπαρχουν ακομα bf16 trainable params: {bad[:3]}"
print("OK - ολα τα trainable params ειναι fp32.")

In [ ]:
train_result = trainer.train()
print(train_result.metrics)

In [ ]:
# Μετρικες αξιολογησης στο val set
print(trainer.evaluate())

## 9. Αποθηκευση adapter

In [ ]:
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("adapter ->", ADAPTER_DIR)
!ls -lh {ADAPTER_DIR}

## 10. Merge adapter → fp16

Απελευθερωνουμε το 4-bit μοντελο, ξαναφορτωνουμε τη βαση σε **fp16 στη CPU** (8B ≈ 16 GB —
χωρανε στη RAM του Kaggle ~29 GB) και κανουμε merge. Το αποτελεσμα ειναι ενα ενιαιο fp16 μοντελο
ετοιμο για GGUF.

In [ ]:
import gc, torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

# απελευθερωση μνημης (αν υπαρχουν ακομα απο το training ασφαλες και μετα απο restart)
for _n in ["trainer", "model", "merged", "merged4"]:
    if _n in globals():
        del globals()[_n]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.float16, low_cpu_mem_usage=True,
)   # χωρις device_map -> CPU
merged = PeftModel.from_pretrained(base, ADAPTER_DIR)
merged = merged.merge_and_unload()
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print("merged fp16 ->", MERGED_DIR)

del base, merged
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()